In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import datetime as dt
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, balanced_accuracy_score, confusion_matrix, precision_recall_curve, roc_auc_score, accuracy_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.impute import KNNImputer
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE


---
---

## 中興Paper 

1. 去掉na值超過1000筆的欄位(columns, 也就是p)
2. knn填補剩下的na, 用n_neighbors=10, weights='distance'
3. 隨機森林特徵選擇前140筆重要值較大的特徵
4. SMOTE over resampling, fail:pass=1:1
5. training, testing split拆8:2再做決策樹(CART) modeling (criterion='gini', max_depth=6)

#### 結果: FAR=21.9% 與paper的結果 FAR=9.22% 相差甚遠

In [65]:
secom = pd.read_csv(r'C:\Users\User\Documents\GitHub\psychic-spoon\DataSet\secom.csv', sep='\t')

X = secom.drop(columns=['Time', 'Pass/Fail'])
y = secom['Pass/Fail']
na_counts = X.isna().sum()
X = X.loc[:, na_counts<=1000].copy()

knn = KNNImputer(n_neighbors=10, weights='distance')
X_imputed = pd.DataFrame(knn.fit_transform(X), columns=X.columns, index=X.index)

rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
rf.fit(X_imputed, y)
important_ = pd.Series(rf.feature_importances_, index=X_imputed.columns)
top140_features = important_.sort_values(ascending=False).head(140).index.tolist()

X_top140 = X_imputed[top140_features].copy()

smote = SMOTE(sampling_strategy=1, k_neighbors=5, random_state=42)

X_balanced, y_balanced = smote.fit_resample(X_top140, y)

X_train, X_test, y_train, y_test = train_test_split(X_balanced, y_balanced, test_size=0.2, random_state=42)

dtc = DecisionTreeClassifier(random_state=42, criterion='gini', max_depth=6)
dtc_model =dtc.fit(X_train, y_train)

dtc_y_pred = dtc_model.predict(X_test)
dtc_y_prob = dtc_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, dtc_y_pred, labels=[1, -1])  # [[TP, FN],[FP, TN]]
cm_df = pd.DataFrame(cm, index=["True 1", "True -1"], columns=["Pred 1", "Pred -1"])
print(cm_df)

TP, FN, FP, TN = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
FAR = FP / (FP + TN) if (FP + TN) > 0 else np.nan
print("FAR = ", round(FAR*100, 2), "%")

         Pred 1  Pred -1
True 1      237       43
True -1      67      239
FAR =  21.9 %


---
---

## 先拆8:2 再training data上做SMOTE

In [9]:
secom = pd.read_csv(r'C:\Users\User\Documents\GitHub\psychic-spoon\DataSet\secom.csv', sep='\t')

X = secom.drop(columns=['Time', 'Pass/Fail'])
y = secom['Pass/Fail']
na_counts = X.isna().sum()
X = X.loc[:, na_counts<=1000].copy()

knn = KNNImputer(n_neighbors=10, weights='distance')
X_imputed = pd.DataFrame(knn.fit_transform(X), columns=X.columns, index=X.index)

rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
rf.fit(X_imputed, y)
important_ = pd.Series(rf.feature_importances_, index=X_imputed.columns)
top140_features = important_.sort_values(ascending=False).head(140).index.tolist()

X_top140 = X_imputed[top140_features].copy()

X_train, X_test, y_train, y_test = train_test_split(X_top140, y, test_size=0.2, stratify=y,random_state=42)

smote = SMOTE(sampling_strategy=1, k_neighbors=5, random_state=42)

X_balanced, y_balanced = smote.fit_resample(X_train, y_train)

dtc = DecisionTreeClassifier(random_state=42)
dtc_model =dtc.fit(X_balanced, y_balanced)

dtc_y_pred = dtc_model.predict(X_test)
dtc_y_prob = dtc_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, dtc_y_pred, labels=[1, -1])  # [[TP, FN],[FP, TN]]
TP, FN, FP, TN = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
FAR = FP / (FP + TN) if (FP + TN) > 0 else np.nan
print("Confusion Matrix:\n", cm)
print("FAR = ", round(FAR*100, 2), "%")

Confusion Matrix:
 [[  6  15]
 [ 31 262]]
FAR =  10.58 %


---
---

In [8]:
secom = pd.read_csv(r'C:\Users\User\Documents\GitHub\psychic-spoon\DataSet\secom.csv', sep='\t')

X = secom.drop(columns=['Time', 'Pass/Fail'])
y = secom['Pass/Fail'].replace({-1:0, 1:1}).astype(int)
na_counts = X.isna().sum()
X = X.loc[:, na_counts<=1000].copy()

knn = KNNImputer(n_neighbors=10, weights='distance')
X_imputed = pd.DataFrame(knn.fit_transform(X), columns=X.columns, index=X.index)

rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
rf.fit(X_imputed, y)
important_ = pd.Series(rf.feature_importances_, index=X_imputed.columns)
top140_features = important_.sort_values(ascending=False).head(140).index.tolist()

X_top140 = X_imputed[top140_features].copy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def compute_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])  # [[TP, FN],[FP, TN]]
    TP, FN, FP, TN = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    acc = accuracy_score(y_true, y_pred)
    sensitivity = TP/(TP+FN) if (TP+FN)>0 else np.nan
    specificity = TN/(TN+FP) if (TN+FP)>0 else np.nan
    FAR = FP/(FP+TN) if (FP+TN)>0 else np.nan
    GM = np.sqrt(sensitivity*specificity) if (not np.isnan(sensitivity) and not np.isnan(specificity)) else np.nan
    return {'acc':acc, 'FAR':FAR, 'sensitivity':sensitivity, 'specificity':specificity, 'GM':GM}

all_metrics = []
for fold, (tr_idx, te_idx) in enumerate(skf.split(X_top140, y), start=1):
    X_tr, X_te = X_top140.iloc[tr_idx], X_top140.iloc[te_idx]
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

    smote = SMOTE(sampling_strategy=1, k_neighbors=5, random_state=42)
    X_balanced, y_balanced = smote.fit_resample(X_tr, y_tr)

    # lr = LinearRegression()
    # lr.fit(X_balanced, y_balanced)
    # y_hat = lr.predict(X_te)
    # y_hat = np.clip(y_hat, 0, 1)
    # y_pred = (y_hat>=0.5).astype(int) # >=0.5 -> 1(fail); <0.5 -> -1(pass)

    # clf = DecisionTreeClassifier(criterion='gini', max_depth=6, random_state=42)
    # rfc = RandomForestClassifier(n_estimators=500, random_state=42)
    xgbc = XGBClassifier(objective='binary:logistic', eval_metric='auc', n_estimators=500, learning_rate=0.1, min_child_weight=5, subsample=0.8, tree_method='hist', n_jobs=-1, random_state=42)
    xgbc.fit(X_balanced, y_balanced)
    y_pred = xgbc.predict(X_te)

    m = compute_metrics(y_te.values, y_pred)
    all_metrics.append(m)

    print(f"fold={fold}, acc={m['acc']:.4f}, FAR={m['FAR']:.4f}")
    print("sensitivy = ", round(m["sensitivity"]*100, 2), "%")
    print("specificity = ", round(m["specificity"]*100, 2), "%")
    print("GM = ", round(m["GM"]*100, 2), "%")
    print("-"*40)

avg = {k: np.nanmean([m[k] for m in all_metrics]) for k in ["acc","FAR","sensitivity","specificity","GM"]}
print("[Mean over 5 folds]")
print(f"acc={avg['acc']:.4f}, FAR={avg['FAR']:.4f}")
print("sensitivy = ", round(avg["sensitivity"]*100, 2), "%")
print("specificity = ", round(avg["specificity"]*100, 2), "%")
print("GM = ", round(avg["GM"]*100, 2), "%")

fold=1, acc=0.8981, FAR=0.8571
sensitivy =  95.22 %
specificity =  14.29 %
GM =  36.88 %
----------------------------------------
fold=2, acc=0.9172, FAR=0.8571
sensitivy =  97.27 %
specificity =  14.29 %
GM =  37.28 %
----------------------------------------
fold=3, acc=0.9042, FAR=0.7500
sensitivy =  94.88 %
specificity =  25.0 %
GM =  48.7 %
----------------------------------------
fold=4, acc=0.9201, FAR=0.9524
sensitivy =  98.29 %
specificity =  4.76 %
GM =  21.63 %
----------------------------------------
fold=5, acc=0.9201, FAR=0.8571
sensitivy =  97.6 %
specificity =  14.29 %
GM =  37.34 %
----------------------------------------
[Mean over 5 folds]
acc=0.9119, FAR=0.8548
sensitivy =  96.65 %
specificity =  14.52 %
GM =  36.37 %
